# Import necessary dependencies

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Load and normalize data

In [2]:
dataset = pd.read_csv('movie_reviews.csv.bz2', compression='bz2')

# take a peek at the data
print(dataset.head())
reviews = np.array(dataset['review'])
sentiments = np.array(dataset['sentiment'])

# build train and test datasets
train_reviews = reviews[:35000]
train_sentiments = sentiments[:35000]
test_reviews = reviews[35000:]
test_sentiments = sentiments[35000:]

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


# Extract features from positive and negative reviews

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pickle import Unpickler

with open('norm_train_reviews.p', 'rb') as train_file:
    norm_train_reviews = Unpickler(train_file).load()

with open('norm_test_reviews.p', 'rb') as test_file:
    norm_test_reviews = Unpickler(test_file).load()

# consolidate all normalized reviews
norm_reviews = norm_train_reviews+norm_test_reviews
# get tf-idf features for only positive reviews
positive_reviews = [review for review, sentiment in zip(norm_reviews, sentiments) if sentiment == 'positive']
ptvf = TfidfVectorizer(use_idf=True, min_df=0.02, max_df=0.75, ngram_range=(1, 2), sublinear_tf=True)
ptvf_features = ptvf.fit_transform(positive_reviews)
# get tf-idf features for only negative reviews
negative_reviews = [review for review, sentiment in zip(norm_reviews, sentiments) if sentiment == 'negative']
ntvf = TfidfVectorizer(use_idf=True, min_df=0.02, max_df=0.75, ngram_range=(1, 2), sublinear_tf=True)
ntvf_features = ntvf.fit_transform(negative_reviews)
# view feature set dimensions
print(ptvf_features.shape, ntvf_features.shape)

(25000, 927) (25000, 925)


# Topic Modeling on Reviews

In [4]:
import pyLDAvis.lda_model
from sklearn.decomposition import NMF
# import topic_model_utils as tmu

pyLDAvis.enable_notebook()
total_topics = 10

## Display and visualize topics for positive reviews

In [10]:
# build topic model on positive sentiment review features
pos_nmf = NMF(n_components=total_topics, solver='cd', max_iter=500,
               random_state=42, l1_ratio=.85) # alpha_W=.1, alpha_H=.1,
pos_nmf.fit(ptvf_features)      
# extract features and component weights
pos_feature_names = np.array(ptvf.get_feature_names_out())
pos_weights = pos_nmf.components_
# extract and display topics and their components
pos_feature_names = np.array(ptvf.get_feature_names_out())
feature_idxs = np.argsort(-pos_weights)[:, :15]
topics = [pos_feature_names[idx] for idx in feature_idxs]
for idx, topic in enumerate(topics):
    print('Topic #'+str(idx+1)+':')
    print(', '.join(topic))
    print()

Topic #1:
like, really, think, but, get, say, go, know, would, thing, bad, good, lot, much, look

Topic #2:
movie, watch, great, love, see movie, movie not, see, watch movie, enjoy, story, great movie, love movie, recommend, think, like

Topic #3:
show, episode, series, season, tv, watch, television, character, great, air, first, new, every, love, hope

Topic #4:
life, family, man, young, love, woman, father, mother, friend, live, girl, child, go, son, wife

Topic #5:
play, performance, role, actor, cast, good, well, great, star, excellent, also, support, give, john, character

Topic #6:
film, see, see film, film not, great, watch, recommend, watch film, great film, good film, film but, make, film make, film see, make film

Topic #7:
character, scene, story, use, world, one, take, make, no, way, seem, many, work, time, war

Topic #8:
funny, comedy, laugh, hilarious, humor, fun, joke, moment, time, but, line, comic, romantic, scene, make

Topic #9:
ever, good, one, ever see, one good, m

In [11]:
pyLDAvis.lda_model.prepare(pos_nmf, ptvf_features, ptvf, mds='mmds')

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
6      0.055021  0.046645       1        1  16.699296
1     -0.045547 -0.290632       2        1  11.253435
3     -0.201557  0.247379       3        1  11.081369
5     -0.186203 -0.256077       4        1  11.076440
0     -0.093026  0.050932       5        1  10.753561
9      0.002495  0.295702       6        1   9.883307
4      0.265167 -0.001798       7        1   9.686556
2     -0.273076 -0.013723       8        1   6.830093
8      0.240599 -0.280784       9        1   6.427855
7      0.236127  0.202356      10        1   6.308087, topic_info=      Term         Freq        Total Category  logprob  loglift
296   film  2245.000000  2245.000000  Default  30.0000  30.0000
508  movie  2134.000000  2134.000000  Default  29.0000  29.0000
251   ever  1099.000000  1099.000000  Default  28.0000  28.0000
331  funny   916.000000   916.000000  Default  27.0000  27.0000
725   show  1063.000000  1063.000000  Default  26.0000  26.0000
..     ...          ...          ...      ...      ...      ...
853    two    54.600998   400.152973  Topic10  -5.3021   0.7715
254  every    53.225297   467.005078  Topic10  -5.3277   0.5915
885    way    52.310613   463.999843  Topic10  -5.3450   0.5807
306   find    50.895794   517.257088  Topic10  -5.3724   0.4446
476   many    51.244899   585.650533  Topic10  -5.3656   0.3272

[513 rows x 6 columns], token_table=      Topic      Freq   Term
term                        
5         1  0.128335    act
5         2  0.263087    act
5         3  0.038501    act
5         4  0.118710    act
5         5  0.060959    act
...     ...       ...    ...
924      10  0.112779    yet
926       3  0.703607  young
926       4  0.030233  young
926       6  0.098945  young
926       7  0.167656  young

[1918 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[7, 2, 4, 6, 1, 10, 5, 3, 9, 8])

## Display and visualize topics for negative reviews

In [13]:
# build topic model on negative sentiment review features
neg_nmf = NMF(n_components=total_topics, solver='cd', max_iter=500,
              random_state=42, l1_ratio=.85) # alpha_W=.1, alpha_H=.1,
neg_nmf.fit(ntvf_features)      
# extract features and component weights
neg_feature_names = ntvf.get_feature_names_out()
neg_weights = neg_nmf.components_
# extract and display topics and their components
neg_feature_names = np.array(ntvf.get_feature_names_out())
feature_idxs = np.argsort(-neg_weights)[:, :15]
topics = [neg_feature_names[idx] for idx in feature_idxs]
for idx, topic in enumerate(topics):
    print('Topic #'+str(idx+1)+':')
    print(', '.join(topic))
    print()

Topic #1:
get, go, kill, man, guy, take, woman, back, one, end, around, know, come, scene, run

Topic #2:
bad, ever, bad movie, ever see, movie ever, see, movie, one bad, one, ever make, acting, film ever, bad film, movie bad, horrible

Topic #3:
film, make, film not, see, would, but, film but, see film, bad film, watch film, make film, director, watch, say, think

Topic #4:
effect, special, horror, budget, special effect, low, low budget, look, acting, look like, bad, gore, horror movie, flick, like

Topic #5:
movie, watch, think, would, but, like, see, movie not, really, could, good, make, say, watch movie, want

Topic #6:
funny, comedy, laugh, joke, not funny, humor, try, but, stupid, fun, suppose, hilarious, even, but not, good

Topic #7:
character, story, but, actor, seem, play, good, much, performance, role, well, cast, plot, scene, work

Topic #8:
waste, waste time, time, not waste, money, watch, even, hour, not even, plot, spend, movie, acting, crap, complete

Topic #9:
show, e

In [14]:
pyLDAvis.lda_model.prepare(neg_nmf, ntvf_features, ntvf, mds='mmds')

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
6      0.103666  0.123714       1        1  15.719797
4     -0.027070 -0.234356       2        1  15.057029
0      0.159797 -0.002376       3        1  12.527811
2     -0.030336 -0.065270       4        1  10.957949
8      0.110751 -0.145470       5        1   9.315976
3     -0.139710  0.209542       6        1   8.712904
9      0.353303 -0.006055       7        1   8.134848
1     -0.333541  0.035898       8        1   7.129520
5      0.061413  0.292130       9        1   6.236457
7     -0.258272 -0.207757      10        1   6.207708, topic_info=           Term         Freq        Total Category  logprob  loglift
876       waste  1064.000000  1064.000000  Default  30.0000  30.0000
300        film  1382.000000  1382.000000  Default  29.0000  29.0000
332       funny   941.000000   941.000000  Default  28.0000  28.0000
732        show   988.000000   988.000000  Default  27.0000  27.0000
877  waste time   796.000000   796.000000  Default  26.0000  26.0000
..          ...          ...          ...      ...      ...      ...
565     nothing    76.274426   457.784726  Topic10  -4.9743   0.9873
532          no    93.640611   751.188285  Topic10  -4.7692   0.6972
51        awful    74.720421   448.098871  Topic10  -4.9949   0.9881
502       movie   113.047229  1778.417067  Topic10  -4.5808   0.0237
7         actor    71.852165   566.885174  Topic10  -5.0340   0.7138

[524 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
1         1  0.045716  absolutely
1         2  0.144766  absolutely
1         3  0.034287  absolutely
1         4  0.118099  absolutely
1         5  0.095241  absolutely
...     ...       ...         ...
923       3  0.094337       young
923       7  0.781646       young
924       3  0.183104      zombie
924       6  0.785823      zombie
924       8  0.030517      zombie

[2097 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[7, 5, 1, 3, 9, 4, 10, 2, 6, 8])